In [ ]:
# Standard libraries
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Scikit Learn
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier, AdaBoostClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, make_scorer
from sklearn.svm import SVC
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import RandomizedSearchCV
from scipy.stats import loguniform

# MiniLearn — your from-scratch library
# from minilearn.classifiers import LogisticRegression, KNN, GaussianNaiveBayes, DecisionTreeClassifier
from minilearn.preprocessing import StandardScaler, train_test_split
from minilearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
from minilearn.model_selection import KFold


In [2]:
# load processed data
metadata = pd.read_csv("/home/westley/Documents/classes/spring_2026/machine_learning/Project_CSE432-532/SER_Project/processed_data/ravdess_metadata.csv")
features = pd.read_csv("/home/westley/Documents/classes/spring_2026/machine_learning/Project_CSE432-532/SER_Project/processed_data/ravdess_features.csv")

# load dataframes
df = metadata.merge(features, on="filename", how="inner")
features = features.drop(columns=["filename"])

In [3]:
# Baseline model
X = features.copy()
y = df["emotion"]

models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000, random_state=42))]),

    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=30))]),

    "GNB": Pipeline([("scaler", StandardScaler()), ("model", GaussianNB())]),

    "SVC": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", gamma=.01, C=11, random_state=42))]),

    "Decision Tree": DecisionTreeClassifier(max_depth=11, min_samples_leaf=10, min_samples_split=51, random_state=42),

    "Random Forest": RandomForestClassifier(n_estimators=175, criterion="log_loss", random_state=42),

    "Ada Boost": AdaBoostClassifier(n_estimators=175, estimator=DecisionTreeClassifier(max_depth=11, min_samples_leaf=10, min_samples_split=51, random_state=42), learning_rate=.3, random_state=42)
}

cv = KFold(n_splits=5)

results=[]
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
    results.append({'Model': name, 'Mean Accuracy': scores.mean(), 'Std Dev': scores.std()})

comparison_df = pd.DataFrame(results).sort_values(by='Mean Accuracy', ascending=False)
print(comparison_df)

                 Model  Mean Accuracy   Std Dev
6            Ada Boost       0.490702  0.019443
0  Logistic Regression       0.489204  0.017737
3                  SVC       0.467587  0.028831
5        Random Forest       0.454107  0.034295
1                  KNN       0.391505  0.011172
4        Decision Tree       0.327846  0.024345
2                  GNB       0.243303  0.044743


In [4]:
from sklearn.model_selection import KFold, cross_val_score
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
# Baseline model
X = features.copy()
y = df["emotion"]

models = {
    "Logistic Regression": Pipeline([("scaler", StandardScaler()), ("model", LogisticRegression(max_iter=2000, random_state=42))]),

    "KNN": Pipeline([("scaler", StandardScaler()), ("model", KNeighborsClassifier(n_neighbors=30))]),

    "GNB": Pipeline([("scaler", StandardScaler()), ("model", GaussianNB())]),

    "SVC": Pipeline([("scaler", StandardScaler()), ("model", SVC(kernel="rbf", gamma=.01, C=11, random_state=42))]),

    "Decision Tree": DecisionTreeClassifier(max_depth=11, min_samples_leaf=10, min_samples_split=51, random_state=42),

    "Random Forest": RandomForestClassifier(n_estimators=175, criterion="log_loss", random_state=42),

    "Ada Boost": AdaBoostClassifier(n_estimators=175, estimator=DecisionTreeClassifier(max_depth=11, min_samples_leaf=10, min_samples_split=51, random_state=42), learning_rate=.3, random_state=42)
}

cv = KFold(n_splits=5)

results=[]
for name, model in models.items():
    scores = cross_val_score(model, X, y, cv=cv, scoring="f1_macro")
    results.append({'Model': name, 'Mean Accuracy': scores.mean(), 'Std Dev': scores.std()})

comparison_df = pd.DataFrame(results).sort_values(by='Mean Accuracy', ascending=False)
print(comparison_df)

                 Model  Mean Accuracy   Std Dev
6            Ada Boost       0.490702  0.019443
0  Logistic Regression       0.489204  0.017737
3                  SVC       0.467587  0.028831
5        Random Forest       0.454107  0.034295
1                  KNN       0.391505  0.011172
4        Decision Tree       0.327846  0.024345
2                  GNB       0.243303  0.044743


In [ ]:
pipeline = Pipeline([("scaler", StandardScaler()), ("model", SVC())])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)

parameter_distributions = {"model__C": loguniform(1e0, 1e2), "model__gamma": loguniform(1e-4, 1e0), "model__kernel": ["rbf"]}

random_search = RandomizedSearchCV(estimator=pipeline, param_distributions=parameter_distributions, scoring="f1_macro", n_jobs=-1, cv=5, n_iter=50, random_state=42)

random_search.fit(X_train, y_train)

print(random_search.best_params_)
print(random_search.best_score_)

best_model = random_search.best_estimator_

y_pred = best_model.predict(X_test)

print("Macro F1:", f1_score(y_test, y_pred, average="macro"))
print("Accuracy:", accuracy_score(y_test, y_pred))
print(confusion_matrix(y_test, y_pred))

{'model__C': np.float64(45.44383960336016), 'model__gamma': np.float64(0.0026730883107816713), 'model__kernel': 'rbf'}
0.6853716636926936
Macro F1: 0.6912342207949111
Accuracy: 0.6965376782077393
[[59  0  4  3  4  0  3  2]
 [ 0 60  2  2  5  0  6  0]
 [ 4  2 23  0  0  1  3  6]
 [11  3  2 45  6  1  5  2]
 [ 7  3  0  7 52  2  3  1]
 [ 0  1  0  1  1 32  3  0]
 [ 3  7  3  6  3  1 50  2]
 [ 4  2  5  4  2  0  1 21]]
